# 📖 Notebook 2: News Feed Ranking

A chronological feed is the simplest approach, but it's not always the best experience.  
Users care more about **relevance** than raw time order. This notebook explores how to rank posts.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why chronological feeds don't scale well for engagement
- How to build a simple scoring function for posts
- The concept of **affinity** (how close are two users?)
- How to combine recency, popularity, and affinity into a feed rank

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/fb-news-feed
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import time
import math
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "newsfeed_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Test connection
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker compose up -d")

## 🤔 Why Not Just Sort by Time?

A purely chronological feed has problems:

- Your best friend's post from 2 hours ago gets buried under 50 posts from pages you barely care about
- A viral post with 10,000 likes ranks the same as a post nobody interacted with
- High-frequency posters dominate the feed

Facebook switched from chronological to **ranked** feeds in 2009.  
The result? Users engaged **more** because they saw content they actually cared about.

### The Original Facebook EdgeRank Formula

Facebook's original ranking algorithm was called **EdgeRank**:

```
Score = Affinity × Weight × Decay
```

| Factor | What It Means | Example |
|--------|--------------|--------|
| **Affinity** | How close are you to the author? | You interact with Alice's posts often → high affinity |
| **Weight** | How engaging is this type of content? | Photos > links > plain text |
| **Decay** | How old is the post? | Newer posts score higher |

## Step 1: Add Engagement Data

To rank posts, we need signals. Let's add likes and a simple interaction log to our database.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Create tables for engagement signals
cur.execute("""
    CREATE TABLE IF NOT EXISTS likes (
        user_id INTEGER REFERENCES users(id),
        post_id INTEGER REFERENCES posts(id),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (user_id, post_id)
    )
""")

# Interaction log: tracks when a user interacts with another user's content.
# This is how we measure "affinity" between two users.
cur.execute("""
    CREATE TABLE IF NOT EXISTS interactions (
        id SERIAL PRIMARY KEY,
        actor_id INTEGER REFERENCES users(id),
        target_user_id INTEGER REFERENCES users(id),
        interaction_type VARCHAR(20),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
cur.execute("CREATE INDEX IF NOT EXISTS idx_interactions_actor ON interactions(actor_id, target_user_id)")
conn.commit()
print("✅ Created likes and interactions tables")
conn.close()

# User 1 follows users [11, 25, 35, 37, 41, 44, 51, 52] in our seed data.
# We seed affinity so user 1 interacts a lot with followees 11 and 25,
# weakly with 35 and 37, and with celebrity 51. Followees 41, 44, 52
# receive zero interactions — they should rank lower.
import random
random.seed(42)

conn = get_db()
cur = conn.cursor()

# Generate likes — celebrity posts get more likes than regular posts
cur.execute("SELECT id, author_id FROM posts")
post_rows = cur.fetchall()

like_data = []
for post_id, author_id in post_rows:
    n_likes = random.randint(10, 40) if author_id > 50 else random.randint(0, 10)
    likers = random.sample(range(1, 51), min(n_likes, 50))
    for liker in likers:
        like_data.append((liker, post_id))

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO likes (user_id, post_id) VALUES %s ON CONFLICT DO NOTHING",
    like_data,
)

# Interaction history for user 1 — only interactions with followees matter
# because the ranked feed only considers posts from followees.
interaction_data = []
for _ in range(30):
    interaction_data.append((1, 11, "like"))
    interaction_data.append((1, 25, "like"))
    interaction_data.append((1, 51, "like"))
for _ in range(5):
    interaction_data.append((1, 35, "like"))
    interaction_data.append((1, 37, "like"))

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO interactions (actor_id, target_user_id, interaction_type) VALUES %s",
    interaction_data,
)

conn.commit()
print(f"✅ Seeded {len(like_data)} likes and {len(interaction_data)} interactions")
conn.close()

# Also add a couple of RECENT posts from friends 11 and 25 so they can
# compete with the celebrity in the ranked feed (otherwise their old posts
# get a very low recency score and Alice dominates).
conn = get_db()
cur = conn.cursor()
cur.execute("""
    INSERT INTO posts (author_id, content, created_at) VALUES
      (11, 'Fresh post from best friend user 11 — coffee time ☕', NOW() - INTERVAL '30 minutes'),
      (11, 'Another recent post from user 11 — just finished a project!',    NOW() - INTERVAL '2 hours'),
      (25, 'Hello from user 25! Anyone up for lunch?',                        NOW() - INTERVAL '45 minutes'),
      (25, 'Second post from user 25 — thoughts on the weather?',             NOW() - INTERVAL '3 hours')
    RETURNING id, author_id, created_at
""")
new_posts = cur.fetchall()

# Push these into the precomputed_feed for their followers so the
# non-ranked/chronological query also surfaces them.
for post_id, author_id, created_at in new_posts:
    cur.execute("""
        INSERT INTO precomputed_feed (user_id, post_id, post_author_id, post_created_at)
        SELECT follower_id, %s, %s, %s FROM follows WHERE followee_id = %s
        ON CONFLICT DO NOTHING
    """, (post_id, author_id, created_at, author_id))

# Give these fresh posts a few likes each so popularity isn't zero
for post_id, _a, _c in new_posts:
    cur.execute("""
        INSERT INTO likes (user_id, post_id)
        SELECT generate_series(2, 8), %s
        ON CONFLICT DO NOTHING
    """, (post_id,))

conn.commit()
conn.close()


## Step 2: Build the Scoring Function

Our simple ranking score combines three signals:

```
score = (affinity_score × 3.0) + (popularity_score × 1.0) + (recency_score × 2.0)
```

Each component is normalised to a 0–1 range so we can combine them with weights.

In [ ]:
def compute_affinity(user_id: int, author_id: int, conn) -> float:
    """
    How much does this user interact with the author?
    Returns a value between 0 and 1.
    
    Higher = the user likes/comments on this author's posts frequently.
    """
    cur = conn.cursor()
    cur.execute(
        "SELECT COUNT(*) FROM interactions WHERE actor_id = %s AND target_user_id = %s",
        (user_id, author_id)
    )
    count = cur.fetchone()[0]
    # Use logarithmic scaling so it doesn't grow linearly forever
    # 0 interactions → 0.0, 10 interactions → 0.7, 30+ → ~1.0
    return min(1.0, math.log(count + 1) / math.log(31))


def compute_popularity(post_id: int, conn) -> float:
    """
    How popular is this post? (based on likes)
    Returns a value between 0 and 1.
    """
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM likes WHERE post_id = %s", (post_id,))
    likes = cur.fetchone()[0]
    # 0 likes → 0.0, 10 likes → 0.65, 40+ → 1.0
    return min(1.0, math.log(likes + 1) / math.log(41))


def compute_recency(post_created_at: datetime) -> float:
    """
    How recent is this post?
    Returns a value between 0 and 1.
    
    Just posted → 1.0, 1 day ago → 0.5, 1 week ago → ~0.0
    """
    age_hours = (datetime.now() - post_created_at).total_seconds() / 3600
    # Exponential decay: halves every 24 hours
    return math.exp(-0.029 * age_hours)  # 0.029 ≈ ln(2)/24


# Demo: show scores for a few posts from user 1's perspective
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT p.id, p.content, p.created_at, p.author_id, u.display_name
    FROM posts p
    JOIN users u ON u.id = p.author_id
    WHERE p.author_id IN (11, 25, 35, 41, 51)
    ORDER BY p.created_at DESC
    LIMIT 8
""")
sample_posts = cur.fetchall()

print("📊 Scoring components for user 1's feed:")
print(f"{'Post':>5}  {'Author':<20}  {'Affinity':>8}  {'Popular':>8}  {'Recency':>8}")
print("-" * 60)

for post in sample_posts:
    aff = compute_affinity(1, post["author_id"], conn)
    pop = compute_popularity(post["id"], conn)
    rec = compute_recency(post["created_at"])
    print(f"{post['id']:>5}  {post['display_name']:<20}  {aff:>8.2f}  {pop:>8.2f}  {rec:>8.2f}")

print("\n💡 User 1 interacts a lot with followees 11, 25, and celebrity 51 → high affinity")
conn.close()


## Step 3: Ranked Feed

Now let's build a function that fetches a feed and ranks it by our combined score.

In [ ]:
# Weights for each signal — tune these to change ranking behaviour
WEIGHT_AFFINITY  = 3.0  # how much we value "closeness" to the author
WEIGHT_POPULARITY = 1.0  # how much we value raw likes
WEIGHT_RECENCY   = 2.0  # how much we value freshness


def ranked_feed(user_id: int, limit: int = 20) -> list:
    """
    Get a ranked feed for the user.
    
    1. Fetch candidate posts from people the user follows (recent ones)
    2. Score each post
    3. Sort by score descending
    4. Return top N
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    # Get candidate posts (last ~200 posts from followees)
    cur.execute("""
        SELECT p.id, p.content, p.created_at, p.author_id,
               u.username, u.display_name
        FROM posts p
        JOIN users u ON u.id = p.author_id
        WHERE p.author_id IN (
            SELECT followee_id FROM follows WHERE follower_id = %s
        )
        ORDER BY p.created_at DESC
        LIMIT 200
    """, (user_id,))
    candidates = cur.fetchall()
    
    # Score each post
    scored = []
    for post in candidates:
        affinity  = compute_affinity(user_id, post["author_id"], conn)
        popularity = compute_popularity(post["id"], conn)
        recency   = compute_recency(post["created_at"])
        
        score = (
            WEIGHT_AFFINITY  * affinity +
            WEIGHT_POPULARITY * popularity +
            WEIGHT_RECENCY   * recency
        )
        scored.append({**post, "score": score,
                       "_aff": affinity, "_pop": popularity, "_rec": recency})
    
    # Sort by score (highest first)
    scored.sort(key=lambda x: x["score"], reverse=True)
    conn.close()
    
    return scored[:limit]


# Show the ranked feed for user 1
feed = ranked_feed(user_id=1, limit=15)

print("📰 Ranked feed for user 1:")
print(f"{'Rank':>4}  {'Score':>6}  {'Author':<22}  {'Aff':>4} {'Pop':>4} {'Rec':>4}  Post")
print("-" * 90)
for i, post in enumerate(feed, 1):
    print(
        f"{i:>4}  {post['score']:>6.2f}  {post['display_name']:<22}  "
        f"{post['_aff']:.1f}  {post['_pop']:.1f}  {post['_rec']:.1f}   "
        f"{post['content'][:35]}"
    )

print("\n💡 Posts from followees with high affinity (11, 25, celebrity 51) rank higher!")


---
## 🐌 The N+1 Problem Hiding in `ranked_feed`

Look again at the scoring loop. For **every candidate post** it calls
`compute_affinity` (one query) and `compute_popularity` (another query). With 200
candidates that's **400 round trips to Postgres to render one feed**.

This is the single most common way a ranked feed falls over in production, and
it doesn't show up in a unit test — only under load. Let's measure it, then fix
it, then measure again.

In [ ]:
# ── Measure the naive version ───────────────────────────────────────────
t0 = time.perf_counter()
naive = ranked_feed(user_id=1, limit=15)
naive_ms = (time.perf_counter() - t0) * 1000

conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT COUNT(*) FROM posts
    WHERE author_id IN (SELECT followee_id FROM follows WHERE follower_id = 1)
""")
candidate_count = min(cur.fetchone()[0], 200)
conn.close()
print(f"🐌 Naive ranked_feed: {naive_ms:,.1f} ms "
      f"({candidate_count} candidates x 2 queries = ~{candidate_count * 2} round trips)")


# ── The fix: fetch every signal in ONE query, score in memory ───────────
def ranked_feed_batched(user_id: int, limit: int = 20) -> list:
    """Same ranking, but the database is asked once instead of 2N+1 times.

    Affinity and like counts are aggregated server-side and joined onto the
    candidate set. Scoring itself is pure arithmetic — do it in the app.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        WITH followees AS (
            SELECT followee_id FROM follows WHERE follower_id = %(uid)s
        ),
        candidates AS (
            SELECT p.id, p.content, p.created_at, p.author_id
            FROM posts p
            WHERE p.author_id IN (SELECT followee_id FROM followees)
            ORDER BY p.created_at DESC
            LIMIT 200
        ),
        affinity AS (
            SELECT target_user_id, COUNT(*) AS n
            FROM interactions
            WHERE actor_id = %(uid)s
            GROUP BY target_user_id
        ),
        popularity AS (
            SELECT post_id, COUNT(*) AS n
            FROM likes
            WHERE post_id IN (SELECT id FROM candidates)
            GROUP BY post_id
        )
        SELECT c.id, c.content, c.created_at, c.author_id,
               u.username, u.display_name,
               COALESCE(a.n, 0) AS interaction_count,
               COALESCE(pop.n, 0) AS like_count
        FROM candidates c
        JOIN users u ON u.id = c.author_id
        LEFT JOIN affinity a ON a.target_user_id = c.author_id
        LEFT JOIN popularity pop ON pop.post_id = c.id
    """, {"uid": user_id})
    rows = cur.fetchall()
    conn.close()

    scored = []
    for row in rows:
        affinity = min(1.0, math.log(row["interaction_count"] + 1) / math.log(31))
        popularity = min(1.0, math.log(row["like_count"] + 1) / math.log(41))
        recency = compute_recency(row["created_at"])
        scored.append({**row, "_aff": affinity, "_pop": popularity, "_rec": recency,
                       "score": (WEIGHT_AFFINITY * affinity
                                 + WEIGHT_POPULARITY * popularity
                                 + WEIGHT_RECENCY * recency)})
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:limit]


t0 = time.perf_counter()
batched = ranked_feed_batched(user_id=1, limit=15)
batched_ms = (time.perf_counter() - t0) * 1000
print(f"⚡ Batched ranked_feed: {batched_ms:,.1f} ms (1 query)")
print(f"   Speedup: {naive_ms / batched_ms:.1f}x")

# Same ranking, or the optimisation isn't an optimisation.
assert [p["id"] for p in naive] == [p["id"] for p in batched], "ranking changed!"
print("✅ Identical ordering — this is a pure performance fix, not a behaviour change.")
print()
print("💡 The trade-off: the batched query is harder to read and pins the scoring")
print("   inputs to what SQL can aggregate. Real ranking systems go further and")
print("   move signals into a feature store precisely so the scorer never has to")
print("   ask the primary database anything at request time.")

---
## 📄 Paginating a Ranked Feed (Harder Than It Looks)

Notebook 1 and Notebook 4 use cursor pagination over **time**, which works
because a post's timestamp never changes. A ranked feed has no such luxury:
`score` depends on **recency**, so every post's score drops continuously, and it
depends on likes, which change while the user is scrolling.

Cursor-on-score therefore does *not* give you a stable page boundary. Watch it
break.

In [ ]:
# Page 1, then the world changes, then page 2 — using score as the cursor.

def ranked_page_by_score(user_id, max_score=None, limit=5):
    """Cursor pagination where the cursor is the score of the last post seen."""
    all_scored = ranked_feed_batched(user_id, limit=200)
    if max_score is not None:
        all_scored = [p for p in all_scored if p["score"] < max_score]
    page = all_scored[:limit]
    return page, (page[-1]["score"] if page else None)


full_ranking = ranked_feed_batched(1, limit=200)
page1 = full_ranking[:5]
cursor = page1[-1]["score"]
print(f"Page 1 post ids: {[p['id'] for p in page1]}   cursor score = {cursor:.4f}")

# Pick the post sitting just below the page boundary whose author the user has
# never interacted with. Then the user does the most ordinary thing possible:
# they like something else from that author, so affinity jumps.
below = next(p for p in full_ranking[5:] if p["_aff"] == 0.0)
print(f"Just below the fold: post {below['id']} by @{below['username']} "
      f"(score {below['score']:.4f}, affinity 0.0)")

conn = get_db()
conn.autocommit = True
cur = conn.cursor()
psycopg2.extras.execute_values(
    cur,
    "INSERT INTO interactions (actor_id, target_user_id, interaction_type) VALUES %s",
    [(1, below["author_id"], "like")] * 30,
)
conn.close()
print(f"👍 User 1 interacts 30x with @{below['username']} — affinity for ALL their posts rises.")

# The user scrolls. Page 2 = everything scoring below the cursor we handed out.
page2, _ = ranked_page_by_score(1, max_score=cursor, limit=5)
print(f"Page 2 post ids: {[p['id'] for p in page2]}")

seen = {p["id"] for p in page1} | {p["id"] for p in page2}
duplicates = {p["id"] for p in page1} & {p["id"] for p in page2}
print()
glitched = False
if below["id"] not in seen:
    glitched = True
    print(f"❌ SKIPPED: post {below['id']} was never shown. Its score rose above the")
    print("   cursor we already handed out, so page 2 filtered it away — and page 1")
    print("   had already been delivered. The user silently loses it.")
if duplicates:
    glitched = True
    print(f"❌ DUPLICATED: post(s) {sorted(duplicates)} appear on BOTH pages. Recency")
    print("   decay alone nudged their score below the cursor between requests, so")
    print("   they slid forward into the next page.")
if not glitched:
    print("⚠️  No visible glitch this run, but the boundary still moved. A score")
    print("   cursor is a moving target by construction.")
print()
print("Note that BOTH failures come from the same cause and neither needs unusual")
print("traffic: recency decay guarantees every score drifts between requests.")
assert glitched, "expected the ranked cursor to glitch"

print()
print("Ways out, and what each costs:")
print("  1. Snapshot the ranked id list on page 1 and page through the frozen")
print("     list.  Cost: the feed stops updating mid-scroll, plus per-session")
print("     storage for the list.  This is what most real feeds do.")
print("  2. Page on (score, post_id) with scores frozen at request time.")
print("     Cost: the same snapshot, just implicit rather than explicit.")
print("  3. Send already-seen ids back with each request and filter them out.")
print("     Cost: the list grows without bound on a long scroll, and it fixes")
print("     duplicates but NOT skipped posts — the failure we just produced.")
print()
print("💡 Interview-grade answer: 'a ranked feed is paginated over a snapshot,")
print("   not over live scores — otherwise the page boundary moves under you.'")

## Step 4: Chronological vs Ranked — A Comparison

In [ ]:
# Get chronological feed for comparison
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT p.id, p.content, p.created_at, p.author_id,
           u.username, u.display_name
    FROM posts p
    JOIN users u ON u.id = p.author_id
    WHERE p.author_id IN (
        SELECT followee_id FROM follows WHERE follower_id = 1
    )
    ORDER BY p.created_at DESC
    LIMIT 10
""", )
chrono_feed = cur.fetchall()
conn.close()

ranked = ranked_feed(user_id=1, limit=10)

print("📅 Chronological Feed (newest first):")
print("-" * 60)
for post in chrono_feed:
    print(f"  [{post['created_at']:%m-%d %H:%M}] @{post['username']}: {post['content'][:40]}")

print("\n🏆 Ranked Feed (most relevant first):")
print("-" * 60)
for post in ranked:
    print(f"  [score={post['score']:.1f}] @{post['username']}: {post['content'][:40]}")

print("\n💡 Notice how the ranked feed surfaces posts from users you interact with most,")
print("   even if they're not the absolute newest posts.")

## 🎛️ Experiment: Tuning the Weights

Try changing the weights below to see how the feed changes.  
In production, these weights would be learned from user behaviour using machine learning.

In [ ]:
# ✏️ EXPERIMENT: Change these weights and re-run this cell!
WEIGHT_AFFINITY   = 3.0   # try 0.0 to ignore affinity
WEIGHT_POPULARITY = 1.0   # try 5.0 to make viral posts dominate
WEIGHT_RECENCY    = 2.0   # try 10.0 to make it nearly chronological

feed = ranked_feed(user_id=1, limit=10)

print(f"🎛️  Weights: affinity={WEIGHT_AFFINITY}, popularity={WEIGHT_POPULARITY}, recency={WEIGHT_RECENCY}")
print()
for i, post in enumerate(feed, 1):
    print(
        f"  {i:>2}. [score={post['score']:.2f}] @{post['username']:<15} "
        f"aff={post['_aff']:.1f} pop={post['_pop']:.1f} rec={post['_rec']:.1f}"
    )

## 🧹 Cleanup

In [ ]:
conn = get_db()
cur = conn.cursor()
# Drop engagement tables first — they reference posts via FK so they would
# block any DELETE on posts otherwise.
cur.execute("DROP TABLE IF EXISTS likes CASCADE")
cur.execute("DROP TABLE IF EXISTS interactions CASCADE")
# Now we can safely remove the demo posts we added for ranking.
cur.execute("DELETE FROM precomputed_feed WHERE post_id IN (SELECT id FROM posts WHERE content LIKE 'Fresh post from best friend%' OR content LIKE 'Another recent post from user 11%' OR content LIKE 'Hello from user 25%' OR content LIKE 'Second post from user 25%')")
cur.execute("DELETE FROM posts WHERE content LIKE 'Fresh post from best friend%' OR content LIKE 'Another recent post from user 11%' OR content LIKE 'Hello from user 25%' OR content LIKE 'Second post from user 25%'")
conn.commit()
print("🧹 Cleaned up likes, interactions, and demo posts")
conn.close()


## 📚 Summary

### Key Takeaways

1. **Chronological feeds** are simple but don't optimise for what users care about
2. **Ranking** combines affinity, popularity, and recency into a single score
3. **Affinity** measures how much you interact with an author — it's the strongest signal
4. **Weights** control the ranking behaviour; in production, ML models learn optimal weights
5. **EdgeRank** was Facebook's original formula; modern systems use deep learning

### Interview Tips

- Mention ranking as an enhancement after the basic feed design
- The formula `score = affinity × weight × decay` is a great one-liner to drop
- Explain that ranking happens **after** candidate retrieval (first get posts, then score them)
- Note that ranking adds latency — caching ranked feeds is important

### Next Up

In **Notebook 3**, we'll dive into **Social Graph Storage** — how to store follow relationships efficiently and answer queries like "who follows me?" and "mutual friends".